In [16]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
import torch
from torch.utils.data import DataLoader

import models

In [2]:
TICKER      = "AAPL"
START_DATE  = "2015-01-01"
END_DATE    = "2024-01-01"
WINDOW_SIZE = 30
BATCH_SIZE  = 64
N_SPLITS    = 5 

In [3]:
def download_data(ticker: str, start: str, end: str) -> pd.DataFrame:
    """Download OHLCV data from Yahoo Finance."""
    df = yf.download(ticker, start=start, end=end, auto_adjust=True)
    df.dropna(how="all", inplace=True)          # drop fully-empty rows
    print(f"Downloaded {len(df)} rows for {ticker}")
    return df

In [ ]:
data = download_data(TICKER, START_DATE, END_DATE)
data

[*********************100%***********************]  1 of 1 completed

Downloaded 2264 rows for AAPL


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2015-01-02,24.192612,24.659514,23.754475,24.648449,212818400
2015-01-05,23.511072,24.042146,23.325197,23.962485,257142000
2015-01-06,23.513269,23.772167,23.152581,23.575228,263188400
2015-01-07,23.842979,23.942555,23.610634,23.721274,160423600
2015-01-08,24.759077,24.816610,24.053192,24.170472,237458000
...,...,...,...,...,...
2023-12-22,191.433090,193.222829,190.810137,192.995392,37149600
2023-12-26,190.889282,191.719877,190.671743,191.443012,28919300


In [5]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Use Close, Volume, and simple technical indicators as model inputs.
    The TARGET is next-day Close price.
    """
    data = pd.DataFrame(index=df.index)

    close = df["Close"].squeeze()
    volume = df["Volume"].squeeze()

    data["Close"] = close
    data["Volume"] = volume
    data["Return"] = close.pct_change()           # daily % return
    data["MA7"] = close.rolling(7).mean()      # 7-day moving avg
    data["MA21"] = close.rolling(21).mean()     # 21-day moving avg
    data["Volatility"] = close.rolling(7).std()       # short-term volatility
    data["Momentum"] = close - close.shift(5)       # 5-day price momentum

    # Median imputation for any NaNs produced by rolling calculations
    for col in data.columns:
        if data[col].isna().any():
            data[col] = data[col].fillna(data[col].median())

    return data

In [ ]:
data = build_features(data)
data

In [7]:
def normalize(data: pd.DataFrame):
    """
    MinMaxScaler → each feature scaled to [0, 1].
    We keep the Close scaler separately so we can inverse-transform
    predictions back to real dollar values later.
    """
    feature_scaler = MinMaxScaler()
    close_scaler   = MinMaxScaler()

    scaled_features = feature_scaler.fit_transform(data.values)
    close_scaler.fit(data[["Close"]].values)   # only fit on Close column

    return scaled_features, feature_scaler, close_scaler

In [ ]:
scaled, f_scaler, c_scaler = normalize(data)
scaled, f_scaler, c_scaler

(array([[0.02068562, 0.30214001, 0.52139678, ..., 0.17452833, 0.10914808,
         0.53165452],
        [0.01679837, 0.3730831 , 0.40440088, ..., 0.17452833, 0.10914808,
         0.53165452],
        [0.0168109 , 0.38276079, 0.51816317, ..., 0.17452833, 0.10914808,
         0.53165452],
        ...,
        [0.97202646, 0.03847677, 0.51987086, ..., 0.99824911, 0.17204647,
         0.41949998],
        [0.97445165, 0.01600827, 0.52674768, ..., 0.99912731, 0.16168335,
         0.48904687],
        [0.96852983, 0.02980872, 0.49595555, ..., 1.        , 0.09731385,
         0.46440517]], shape=(2264, 7)),
 MinMaxScaler(),
 MinMaxScaler())

In [9]:
def create_sequences(scaled: np.ndarray, steps: int, close_col_idx: int = 0):
    """
    Convert time series → supervised learning pairs.
      ...
    """
    X, y = [], []
    for i in range(steps, len(scaled)):
        X.append(scaled[i - steps : i])          # past `steps` days
        y.append(scaled[i, close_col_idx])        # next day's Close
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

In [13]:
def split_data(X: np.ndarray, y: np.ndarray, n_splits: int = N_SPLITS):
    # This mirrors real-world forecasting: you never peek at the future.
    tss = TimeSeriesSplit(n_splits=n_splits) # always trains on older data, tests on newer data.
    splits = list(tss.split(X))
    train_idx, test_idx = splits[-1]             # use the final fold (largest possible training set)

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")
    return X_train, X_test, y_train, y_test

In [ ]:
X, y = create_sequences(scaled, WINDOW_SIZE)
X.shape, y.shape

((2234, 30, 7), (2234,))

In [15]:
X_train, X_test, y_train, y_test = split_data(X, y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape 

Train: 1862 samples | Test: 372 samples


((1862, 30, 7), (372, 30, 7), (1862,), (372,))